# TCML RL v2 - SARL + MARL retrain with 1500ep x 5 seeds
Trains both refiners on the same RL windows with much longer
schedule + multi-seed ensemble for inference robustness.
Goal: improve SARL/MARL v1 MAE significantly.

In [ ]:
EPOCHS = 1500
BATCH_SIZE = 16
LR = 3e-4
N_SEEDS = 5

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--index-url', 'https://download.pytorch.org/whl/cu121',
                       'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
AUX_DIR = list(INPUT.rglob('combined_data.csv'))[0].parent
CODE_DIR = list(INPUT.rglob('scripts'))[0].parent
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'TaylorCouetteML'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(CODE_DIR, REPO_DIR)
os.chdir(REPO_DIR)
INPUT_CSV = REPO_DIR / 'data' / 'Input' / 'combined_data.csv'
INPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUX_DIR / 'combined_data.csv', INPUT_CSV)
win_train = REPO_DIR / 'data' / 'rl_windows' / 'rl_switch_windows_train.npz'
win_val   = REPO_DIR / 'data' / 'rl_windows' / 'rl_switch_windows_val.npz'
assert win_train.exists() and win_val.exists()
print('Setup ready, REPO_DIR =', REPO_DIR)

In [ ]:
OUT_SARL = WORK / 'runs' / 'sarl_v2'
OUT_SARL.mkdir(parents=True, exist_ok=True)
args = [sys.executable, 'scripts/train_sarl_v2.py',
        '--windows_train', str(win_train),
        '--windows_val', str(win_val),
        '--epochs', str(EPOCHS),
        '--batch', str(BATCH_SIZE),
        '--lr', str(LR),
        '--n_seeds', str(N_SEEDS),
        '--out_dir', str(OUT_SARL)]
print('>>> SARL v2:', ' '.join(args))
t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO_DIR))
print(f'<<< SARL exit={rc} elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'SARL training failed'

In [ ]:
OUT_MARL = WORK / 'runs' / 'marl_v2'
OUT_MARL.mkdir(parents=True, exist_ok=True)
args = [sys.executable, 'scripts/train_marl_v2.py',
        '--windows_train', str(win_train),
        '--windows_val', str(win_val),
        '--epochs', str(EPOCHS),
        '--batch', str(BATCH_SIZE),
        '--lr', str(LR),
        '--n_seeds', str(N_SEEDS),
        '--out_dir', str(OUT_MARL)]
print('>>> MARL v2:', ' '.join(args))
t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO_DIR))
print(f'<<< MARL exit={rc} elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'MARL training failed'

In [ ]:
for root, dirs, files in os.walk(WORK / 'runs'):
    for f in files:
        p = Path(root) / f
        if p.suffix in ['.pt', '.json', '.npz']:
            print(p.relative_to(WORK), f'({p.stat().st_size/1e6:.2f} MB)')